In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from ydata_profiling import ProfileReport
import webbrowser
import os
from adjustText import adjust_text


# Análisis exploratorio y limpieza de datos

In [2]:
df = pd.read_csv("Titanic.csv") 

# ANÁLISIS EXPLORATORIO DE DATOS (EDA)
df.info() 
df.describe(include='all')
df.head()

profile = ProfileReport(df, title="Perfil de Datos Titanic", explorative=True)
profile.to_file("Titanic_profile.html")

archivo_html = os.path.abspath("Titanic_profile.html")
webbrowser.open(f"file://{archivo_html}")


# LIMPIEZA DE DATOS
df_clean = df.copy()

num_duplicados = df_clean.duplicated().sum()
print(f"Número de filas duplicadas: {num_duplicados}")
if num_duplicados > 0:
    df_clean.drop_duplicates()  

num_duplicados_id = df_clean.duplicated(subset=['PassengerId']).sum()
if num_duplicados_id > 0:
    df_clean.drop_duplicates(subset=['PassengerId'])  

print("\n -------- Valores nulos por columna --------:")
print(df_clean.isnull().sum())

# Eliminar columnas irrelevantes 
eliminar_cols = ["PassengerId", "Name", "Ticket", "Cabin"]
df_clean = df_clean.drop(columns=eliminar_cols)

# Tratamiento valores nulos

if df_clean['Age'].isnull().sum() > 0:
    mediana_edad = df_clean['Age'].median()
    df_clean['Age'] = df_clean['Age'].fillna(mediana_edad)

df_clean["Embarked"] = df_clean["Embarked"].fillna(df["Embarked"].mode()[0])

print(f'\nValores faltantes después de imputar: {df_clean.isna().sum()}')

# Convertir variables categóricas a numéricas con One-Hot Encoding
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True) # Embarked_C, Embarked_Q (binarias 0 = no, 1 = sí). Embarked_S es la categoría base (0 ambas)
df_clean = pd.get_dummies(df_clean, columns=['Sex'], drop_first=True) # 1 = hombre, 0 = mujer (categoría base)

df_clean.info() 
df_clean.describe(include='all')
df_clean.head()

df_clean.to_csv("Titanic_clean.csv", index=False)
df = pd.read_csv("Titanic_clean.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 12/12 [00:00<00:00, 8042.77it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Número de filas duplicadas: 0

 -------- Valores nulos por columna --------:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Valores faltantes después de imputar: Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Survived    891 non-null    int64  
 1   Pclass      891 non-null    int64  
 2   Age         891 non-null    float64
 3   SibSp       891 non-null    int64  
 4   Parch       891 non-null    int64  
 5   Fare        891 non-null    float64
 6   Embarked_Q  891 non-null    bool   
 7   Embarked_S  891 non-null    bool   
 8  

## Matriz de correlación
Antes de aplicar PCA, es útil observar la matriz de correlación para identificar relaciones entre variables y justificar la reducción de dimensionalidad.

In [3]:
plt.figure(figsize=(10,8))
sns.heatmap(df.drop(columns='Survived').corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de correlación entre variables numéricas')
plt.show()

/var/folders/w5/r4237v9175jgsj0grkn8tkhr0000gn/T/ipykernel_76255/288125716.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Análisis de componentes principales (PCA)

In [ ]:
X = df.drop(columns='Survived')
y = df['Survived']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Escalado exitoso. Forma de X_scaled:", X_scaled.shape)

pca = PCA(n_components=X.shape[1])
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X.shape[1])])
df_pca['Survived'] = y.values
df_pca.head()


### Varianza explicada - curva

In [ ]:
# Varianza explicada por componente
var_exp = pca.explained_variance_ratio_
var_exp_acum = np.cumsum(var_exp)

print("\nVarianza explicada por componente:")
for i, v in enumerate(var_exp):
    print(f"PC{i+1}: {v:.4f}")

print(f"\nPC1+PC2: {var_exp_acum[1]*100:.2f}% | PC1+PC2+PC3: {var_exp_acum[2]*100:.2f}%")

n_comp = len(pca.explained_variance_ratio_)

plt.figure(figsize=(8,5))
plt.plot(range(1, n_comp + 1), pca.explained_variance_ratio_,
         marker='o', linewidth=2)

plt.xticks(range(1, n_comp + 1))
plt.xlabel('Componente principal')
plt.ylabel('Varianza explicada')
plt.title('Varianza explicada por cada componente principal')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


### Varianza explicada acumulada - curva

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
plt.xlabel('Número de componentes')
plt.ylabel('Varianza explicada acumulada')
plt.title('Varianza explicada acumulada por PCA')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(np.cumsum(pca.explained_variance_ratio_)*100, marker="o")
plt.xlabel("Número de componentes")
plt.ylabel("% Varianza explicada acumulada")
plt.title(" Porcentaje de varianza explicada por el PCA")
plt.grid(True)
plt.show()

# Biplot

### Biplot: PC1 - PC2

In [ ]:
plt.figure(figsize=(8,6))

# Puntos
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y, palette="Set1", alpha=0.7)

# Vectores
loadings = pca.components_
features = X.columns

scale = 3.0

for i, var in enumerate(features):
    plt.arrow(0, 0, loadings[0, i]*scale, loadings[1, i]*scale, color='black', alpha=0.7, head_width=0.05, linewidth=1.2)
    plt.text(loadings[0, i]*scale*1.15, loadings[1, i]*scale*1.15, var, color='black', fontsize=9, ha='center', va='center')

plt.axhline(0, color='grey', lw=0.6)
plt.axvline(0, color='grey', lw=0.6)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Biplot PCA - PC1 vs PC2")
plt.legend(title="Survived", loc='best')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

### Biplot: PC2 - PC3

In [ ]:
plt.figure(figsize=(8,6))

# Puntos
sns.scatterplot(x=X_pca[:,1], y=X_pca[:,2], hue=y, palette="Set1", alpha=0.7)

# Vectores
loadings = pca.components_
features = X.columns

scale = 3.0

for i, var in enumerate(features):
    plt.arrow(0, 0, loadings[1, i]*scale, loadings[2, i]*scale, color='black', alpha=0.7, head_width=0.05, linewidth=1.2)
    plt.text(loadings[1, i]*scale*1.15, loadings[2, i]*scale*1.15, var, color='black', fontsize=9, ha='center', va='center')

plt.axhline(0, color='grey', lw=0.6)
plt.axvline(0, color='grey', lw=0.6)
plt.xlabel("PC2")
plt.ylabel("PC3")
plt.title("Biplot PCA - PC2 vs PC3")
plt.legend(title="Survived", loc='best')
plt.grid(True, linestyle='--', alpha=0.6)
plt.margins(0.3)
plt.tight_layout()
plt.show()

# Importancia de PC1

### Gráfica de importancia de variables en PC1

In [ ]:
pc1_importancia = pd.Series(pca.components_[0], index=X.columns).sort_values(ascending=False)
pc1_importancia.plot(kind='bar', figsize=(10,6))
plt.title('Importancia de variables en PC1')
plt.ylabel('Peso en PC1')
plt.xlabel('Variable')
plt.grid(True)
plt.show()

A continuación se muestra la importancia de cada variable en la primera componente principal (PC1), lo que permite identificar cuáles influyen más en la variabilidad de los datos.

In [ ]:
pc1_importancia = pd.Series(abs(pca.components_[0]), index=X.columns)
pc1_importancia = pc1_importancia.sort_values(ascending=False)

print("Top variables que más influyen en PC1:")
print(pc1_importancia.head(10))
